In [ ]:


def ejecutar_dax(dax):
    port = get_powerbi_port()

    conn_str = (
        f"Provider=MSOLAP;"
        f"Data Source=localhost:{port};"
    )

    conn = win32com.client.Dispatch("ADODB.Connection")
    conn.Open(conn_str)

    cmd = win32com.client.Dispatch("ADODB.Command")
    cmd.ActiveConnection = conn
    cmd.CommandText = dax

    rs = cmd.Execute()[0]

    columns = [rs.Fields.Item(i).Name for i in range(rs.Fields.Count)]

    rows = []

    while not rs.EOF:
        row = [rs.Fields.Item(i).Value for i in range(rs.Fields.Count)]
        rows.append(row)
        rs.MoveNext()

    rs.Close()
    conn.Close()

    return pd.DataFrame(rows, columns=columns)



In [16]:
tablas = ejecutar_dax("""
SELECT [Name]
FROM $SYSTEM.TMSCHEMA_TABLES
""")




In [20]:
tablas["Name"].tolist()

['Curvas',
 'DateTableTemplate_3bc93af3-2d07-4dc3-b4b3-949de40af15a',
 'Locales_K',
 'Kronos6C',
 'Tiempo',
 'Locales',
 'BQ_v_velocidad_cajero',
 'RutaRGIS',
 'Consolidado',
 'LocalDateTable_4f307101-253e-41cd-ba83-a456d87cc0f5',
 'Locales_Dot',
 'v_velocidad(dia)',
 'LocalDateTable_9b4647fd-c893-43c7-bfc0-3406db3b5262',
 'Tabla fecha',
 'LocalDateTable_51eeb6cc-7b55-4349-8fc7-ac118dad4cb7',
 'Actualización',
 'LocalDateTable_8f324ece-ee0c-4d11-b388-a05e42cd3bca',
 'Orden de Dias',
 'Tabla_MEDIO_PAGO',
 'LOCALES_CON_SCO',
 'Tabla_locales_dotacion',
 'bd_dotacion',
 'Tabla_locales',
 'BQ_trxs_cajas_Sco (2)',
 'D_TIEMPO',
 'Maestro',
 'SP_Metas',
 'BQ_Maestro Locales',
 'LocalDateTable_9aeead67-f7d6-4d6f-8566-d6abeea21b4c',
 'LocalDateTable_e568c6bc-f246-4cc1-879c-a0244a21075c',
 'LocalDateTable_05b44b17-c3e9-49b9-aec4-40920bac04ce',
 'LocalDateTable_001b7a22-7782-4c8a-8688-7cb82d22e740',
 'SP_Busqueda Rapida_Kronos',
 'LocalDateTable_74fd75d3-3443-4dc2-980c-d478792549b4',
 'LocalDateTa

In [22]:
df_productos_digitados = ejecutar_dax("""EVALUATE 'Actualización'""")



In [ ]:
tablas = ejecutar_dax("""
SELECT *
FROM $SYSTEM.TMSCHEMA_TABLES
""")

table_id = tablas.loc[
    tablas["Name"] == "BQ_productos_digitados",
    "ID"
].iloc[0]

print(table_id)

1196317


aqui vale


In [37]:
import re
import subprocess
import pandas as pd
import win32com.client

def get_powerbi_port():
    task_output = subprocess.check_output(
        'tasklist /FI "IMAGENAME eq msmdsrv.exe" /FO CSV',
        shell=True
    ).decode('utf-8', errors='ignore')

    pids = re.findall(r'"msmdsrv\.exe","(\d+)"', task_output)

    if not pids:
        raise Exception("No hay instancia de Power BI abierta")

    pid = pids[-1]

    netstat_output = subprocess.check_output(
        'netstat -ano -p tcp',
        shell=True
    ).decode('utf-8', errors='ignore')

    match = re.search(
        rf'127\.0\.0\.1:(\d+)\s+.*LISTENING\s+{pid}',
        netstat_output
    )

    if not match:
        raise Exception("No se encontró puerto")

    return match.group(1)

In [38]:
get_powerbi_port()

'58881'

In [43]:
df_tablas = ejecutar_dax("""
SELECT *
FROM $SYSTEM.TMSCHEMA_TABLES
""")

print(df_tablas["Name"].tolist())

['Curvas', 'DateTableTemplate_3bc93af3-2d07-4dc3-b4b3-949de40af15a', 'Locales_K', 'Kronos6C', 'Tiempo', 'Locales', 'BQ_v_velocidad_cajero', 'RutaRGIS', 'Consolidado', 'LocalDateTable_4f307101-253e-41cd-ba83-a456d87cc0f5', 'Locales_Dot', 'v_velocidad(dia)', 'LocalDateTable_9b4647fd-c893-43c7-bfc0-3406db3b5262', 'Tabla fecha', 'LocalDateTable_51eeb6cc-7b55-4349-8fc7-ac118dad4cb7', 'Actualización', 'LocalDateTable_8f324ece-ee0c-4d11-b388-a05e42cd3bca', 'Orden de Dias', 'Tabla_MEDIO_PAGO', 'LOCALES_CON_SCO', 'Tabla_locales_dotacion', 'bd_dotacion', 'Tabla_locales', 'BQ_trxs_cajas_Sco (2)', 'D_TIEMPO', 'Maestro', 'SP_Metas', 'BQ_Maestro Locales', 'LocalDateTable_9aeead67-f7d6-4d6f-8566-d6abeea21b4c', 'LocalDateTable_e568c6bc-f246-4cc1-879c-a0244a21075c', 'LocalDateTable_05b44b17-c3e9-49b9-aec4-40920bac04ce', 'LocalDateTable_001b7a22-7782-4c8a-8688-7cb82d22e740', 'SP_Busqueda Rapida_Kronos', 'LocalDateTable_74fd75d3-3443-4dc2-980c-d478792549b4', 'LocalDateTable_73379b11-d4e5-4af5-a368-75cd15

In [ ]:
import pandas as pd
import win32com.client

def ejecutar_dax(dax):

    port = get_powerbi_port()

    conn_str = (
        f"Provider=MSOLAP;"
        f"Data Source=localhost:{port};"
    )

    conn = win32com.client.Dispatch("ADODB.Connection")
    conn.Open(conn_str)

    cmd = win32com.client.Dispatch("ADODB.Command")
    cmd.ActiveConnection = conn
    cmd.CommandText = dax

    rs = cmd.Execute()[0]

    columns = [rs.Fields.Item(i).Name for i in range(rs.Fields.Count)]

    rows = []

    while not rs.EOF:
        rows.append(
            [rs.Fields.Item(i).Value for i in range(rs.Fields.Count)]
        )
        rs.MoveNext()

    rs.Close()
    conn.Close()

    return pd.DataFrame(rows, columns=columns)

df_BQ_productos_digitados = ejecutar_dax("""
EVALUATE 'BQ_productos_digitados'
""")

print(df_BQ_productos_digitados.shape)
print(df_BQ_productos_digitados.head())

com_error: (-2147352567, 'Ocurrió una excepción.', (0, 'Microsoft OLE DB Provider for Analysis Services.', 'The XML for Analysis request timed out before it was completed. Timeout value: 30 sec.', None, 0, -2147467259), None)

In [48]:
import pandas as pd
import win32com.client

def ejecutar_dax(dax):

    port = get_powerbi_port()

    conn_str = (
        f"Provider=MSOLAP;"
        f"Data Source=localhost:{port};"
    )

    conn = win32com.client.Dispatch("ADODB.Connection")
    conn.Open(conn_str)

    cmd = win32com.client.Dispatch("ADODB.Command")
    cmd.ActiveConnection = conn
    cmd.CommandTimeout = 0
    cmd.CommandText = dax

    rs = cmd.Execute()[0]

    columns = [rs.Fields.Item(i).Name for i in range(rs.Fields.Count)]

    rows = []

    while not rs.EOF:

        row = []

        for i in range(rs.Fields.Count):

            valor = rs.Fields.Item(i).Value

            if hasattr(valor, "strftime"):
                valor = valor.strftime("%Y-%m-%d %H:%M:%S")

            row.append(valor)

        rows.append(row)

        rs.MoveNext()

    rs.Close()
    conn.Close()

    return pd.DataFrame(rows, columns=columns)

In [51]:
import os
import gc

os.makedirs("locales", exist_ok=True)

locales = [
    "1198","1199","1593","2638",
    "P001","P010","P037","P048","P052","P059",
    "P061","P064","P066","P068","P075","P080",
    "P081","P084","P089","P096","P102","P104",
    "P105","P106","P107","P108","P109","P110",
    "P112","P114","P115","P117","P118","P119",
    "P120","P122","P123","P125","P126","P130",
    "P132","P133","P134","P135","P138","P139",
    "P141","P142","P143","P144","P145","P146",
    "P147","P156","P158","P159","P172","P177",
    "P180","P181","P191","P192","P193","P195",
    "P198","P199","P201","P202","P203","P206",
    "P207","P208","P215","P216","P219","P220",
    "P221","P223","P224","P225","P226","P227",
    "P229","P231","P236","P238","P258","P259",
    "P261","P262","P264","P291","P333","P422",
    "P465","P516","P723","P724","P769","SO02",
    "SO03","SO04","SO05","SO06","V090","V093",
    "V094","V095","V116","V140","V290","X209","X210"
]

for local in locales:

    try:

        archivo = f"locales/{local}.parquet"

        if os.path.exists(archivo):
            print(f"{local} ya existe. Saltando...")
            continue

        print(f"Descargando {local}...")

        dax = f"""
        EVALUATE
        FILTER(
            'BQ_productos_digitados',
            'BQ_productos_digitados'[codigo_local] = "{local}"
        )
        """

        df_local = ejecutar_dax(dax)

        df_local.to_parquet(
            archivo,
            index=False
        )

        print(f"{local}: {len(df_local):,} filas guardadas")

        del df_local
        gc.collect()

    except Exception as e:

        print(f"ERROR en {local}: {e}")

Descargando 1198...
1198: 50,459 filas guardadas
Descargando 1199...
1199: 126,108 filas guardadas
Descargando 1593...
1593: 130,567 filas guardadas
Descargando 2638...
2638: 59,733 filas guardadas
Descargando P001...
P001: 127,696 filas guardadas
Descargando P010...
P010: 283,610 filas guardadas
Descargando P037...
P037: 112,777 filas guardadas
Descargando P048...
P048: 225,260 filas guardadas
Descargando P052...
P052: 132,929 filas guardadas
Descargando P059...
P059: 125,280 filas guardadas
Descargando P061...
P061: 248,820 filas guardadas
Descargando P064...
P064: 272,096 filas guardadas
Descargando P066...
P066: 128,052 filas guardadas
Descargando P068...
P068: 151,683 filas guardadas
Descargando P075...
P075: 2 filas guardadas
Descargando P080...
P080: 152,721 filas guardadas
Descargando P081...
P081: 246,092 filas guardadas
Descargando P084...
P084: 99,802 filas guardadas
Descargando P089...
P089: 161,622 filas guardadas
Descargando P096...
P096: 124,143 filas guardadas
Descargan